# Runtime Environment Setup

In [32]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB, 580.82.07


In [33]:
import os
os.environ["COMP560_FORCE_REPO_REFRESH"] = "1"


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tarfile

IN_COLAB = 'google.colab' in sys.modules
FORCE_REPO_REFRESH = os.environ.get('COMP560_FORCE_REPO_REFRESH', '0') == '1'
cwd = Path.cwd().resolve()

def unique_paths(paths):
    seen = set()
    result = []
    for path in paths:
        try:
            resolved = path.resolve(strict=False)
        except Exception:
            resolved = path
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            result.append(resolved)
    return result

def existing_unique(paths):
    return [path for path in unique_paths(paths) if path.exists()]

def is_project_dir(path: Path) -> bool:
    return (path / 'train_example.py').exists() and (path / 'evaluate.py').exists()

def has_dataset_files(path: Path) -> bool:
    return (path / 'test.parquet').exists() and (path / 'pairs.parquet').exists()

def maybe_mount_drive():
    if not IN_COLAB:
        return False
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        return True
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print(f'Google Drive mount unavailable: {exc}')
        return False
    return drive_root.exists()

SEARCH_ROOTS = existing_unique([
    cwd,
    *cwd.parents,
    Path('/content'),
    Path('/workspace'),
    Path('/workspaces'),
    Path.home(),
    Path.home() / 'Desktop',
])

DRIVE_MOUNTED = maybe_mount_drive()
DRIVE_ROOT = Path('/content/drive/MyDrive') if DRIVE_MOUNTED else None

PROJECT_DIR_CANDIDATES = []
for root in SEARCH_ROOTS:
    PROJECT_DIR_CANDIDATES.extend([
        root,
        root / 'project-fr',
        root / '560-FaceRecognition' / 'project-fr',
        root / 'Final-Project' / '560-FaceRecognition' / 'project-fr',
    ])
if DRIVE_MOUNTED:
    PROJECT_DIR_CANDIDATES.extend([
        DRIVE_ROOT / '560' / 'Final-Project' / '560-FaceRecognition' / 'project-fr',
        DRIVE_ROOT / '560-FaceRecognition' / 'project-fr',
    ])
PROJECT_DIR_CANDIDATES = unique_paths(PROJECT_DIR_CANDIDATES)

PROJECT_DIR = next((path for path in PROJECT_DIR_CANDIDATES if path.exists() and is_project_dir(path)), None)
clone_target = Path('/content/560-FaceRecognition')
clone_project_dir = clone_target / 'project-fr'

if FORCE_REPO_REFRESH and clone_target.exists():
    print(f'Forcing repo refresh: removing stale clone at {clone_target}')
    shutil.rmtree(clone_target)
    if PROJECT_DIR is not None and str(PROJECT_DIR).startswith(str(clone_project_dir)):
        PROJECT_DIR = None

if PROJECT_DIR is None:
    for search_root in SEARCH_ROOTS + ([DRIVE_ROOT] if DRIVE_MOUNTED else []):
        if search_root is None or not search_root.exists():
            continue
        try:
            match = next(search_root.rglob('train_example.py'), None)
        except Exception:
            match = None
        if match is not None and is_project_dir(match.parent):
            PROJECT_DIR = match.parent.resolve()
            break

if PROJECT_DIR is None:
    if clone_target.exists() and not is_project_dir(clone_project_dir):
        print(f'Removing invalid clone target at {clone_target}')
        shutil.rmtree(clone_target)
    if not is_project_dir(clone_project_dir):
        clone_result = subprocess.run([
            'git',
            'clone',
            'https://github.com/ethan-yountz/560-FaceRecognition',
            str(clone_target),
        ], text=True, capture_output=True, check=False)
        if clone_result.returncode != 0:
            raise RuntimeError(
                'Git clone fallback failed.
'
                'This usually means either the destination already existed in a bad state, or the GitHub repo is not accessible from this notebook runtime.
'
                'If the repo is private, use a Drive-visible repo copy instead of clone fallback.
'
                f'STDOUT:
{clone_result.stdout}
'
                f'STDERR:
{clone_result.stderr}'
            )
    PROJECT_DIR = clone_project_dir.resolve()

REPO_ROOT = PROJECT_DIR.parent

dataset_override = os.environ.get('COMP560_DATASET_DIR')
DATASET_DIR_CANDIDATES = []
if dataset_override:
    override_path = Path(dataset_override).expanduser()
    if ':' in dataset_override and os.name != 'nt':
        print(
            'COMP560_DATASET_DIR is a Windows path, but this notebook kernel is running on Linux. '
            'That path will not be readable unless it is mounted into the runtime.'
        )
    DATASET_DIR_CANDIDATES.extend([override_path, override_path / 'dataset_a'])
DATASET_DIR_CANDIDATES.extend([
    PROJECT_DIR / 'datasets' / 'dataset_a',
    REPO_ROOT / 'datasets' / 'dataset_a',
    cwd / 'dataset_a',
    cwd / 'datasets' / 'dataset_a',
])
if DRIVE_MOUNTED:
    DATASET_DIR_CANDIDATES.extend([
        DRIVE_ROOT / '560' / 'Final-Project' / '560-FaceRecognition' / 'project-fr' / 'datasets' / 'dataset_a',
        DRIVE_ROOT / '560-FaceRecognition' / 'project-fr' / 'datasets' / 'dataset_a',
        DRIVE_ROOT / 'datasets' / 'dataset_a',
    ])
DATASET_DIR_CANDIDATES = unique_paths(DATASET_DIR_CANDIDATES)

DATASET_DIR = next((path for path in DATASET_DIR_CANDIDATES if path.exists() and has_dataset_files(path)), None)

if DATASET_DIR is None:
    dataset_search_roots = existing_unique([
        PROJECT_DIR,
        REPO_ROOT,
        cwd,
        Path('/content'),
        Path('/workspace'),
        Path('/workspaces'),
        Path.home(),
        DRIVE_ROOT if DRIVE_MOUNTED else None,
    ])
    for search_root in dataset_search_roots:
        try:
            match = next(search_root.rglob('dataset_a/test.parquet'), None)
        except Exception:
            match = None
        if match is not None and has_dataset_files(match.parent):
            DATASET_DIR = match.parent.resolve()
            break

if DATASET_DIR is None:
    searched = '\n'.join(str(path) for path in DATASET_DIR_CANDIDATES)
    raise FileNotFoundError(
        'Could not find dataset_a.\n'
        'If this notebook kernel cannot see your local Windows files, put dataset_a in Google Drive and rerun.\n'
        'Or set COMP560_DATASET_DIR to a path that exists inside the notebook runtime.\n'
        f'Checked direct candidates:\n{searched}'
    )

SPLIT_DIR = DATASET_DIR / 'splits' / 'val_15_seed42'
TRAIN_METADATA_PATH = SPLIT_DIR / 'train_metadata.parquet'
VAL_METADATA_PATH = SPLIT_DIR / 'val_metadata.parquet'
VAL_PAIRS_PATH = SPLIT_DIR / 'val_pairs.parquet'

missing_split_files = [
    str(path) for path in (TRAIN_METADATA_PATH, VAL_METADATA_PATH, VAL_PAIRS_PATH) if not path.exists()
]
if missing_split_files:
    split_script = PROJECT_DIR / 'make_validation_split.py'
    if not split_script.exists():
        raise FileNotFoundError(
            'Validation split files are missing, and make_validation_split.py is not available. Missing:\n'
            + '\n'.join(missing_split_files)
        )
    subprocess.run([
        sys.executable,
        str(split_script),
        '--dataset_root', str(DATASET_DIR),
        '--val_fraction', '0.15',
        '--seed', '42',
    ], check=True, cwd=str(PROJECT_DIR))
    missing_split_files = [
        str(path) for path in (TRAIN_METADATA_PATH, VAL_METADATA_PATH, VAL_PAIRS_PATH) if not path.exists()
    ]
    if missing_split_files:
        raise FileNotFoundError(
            'Validation split generation ran, but files are still missing:\n'
            + '\n'.join(missing_split_files)
        )

TRAIN_DATASET_DIR = DATASET_DIR
IMAGES_ARCHIVE = DATASET_DIR / 'images.tar.gz'
IMAGES_DIR = DATASET_DIR / 'images'
needs_extract = (not IMAGES_DIR.exists()) or (not any(IMAGES_DIR.iterdir()))
if needs_extract:
    if not IMAGES_ARCHIVE.exists():
        raise FileNotFoundError(
            f'Images directory is missing and archive not found at {IMAGES_ARCHIVE}'
        )
    with tarfile.open(IMAGES_ARCHIVE) as tar:
        tar.extractall(DATASET_DIR, filter='data')

SAVES_DIR = PROJECT_DIR / 'saves'
if DRIVE_MOUNTED and str(PROJECT_DIR).startswith('/content/') and not str(PROJECT_DIR).startswith('/content/drive/'):
    SAVES_DIR = DRIVE_ROOT / '560' / 'Final-Project' / '560-FaceRecognition' / 'project-fr' / 'saves'

SAVES_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

repo_commit = None
try:
    repo_commit = subprocess.run(
        ['git', 'rev-parse', '--short', 'HEAD'],
        text=True,
        capture_output=True,
        cwd=str(PROJECT_DIR),
        check=False,
    ).stdout.strip() or None
except Exception:
    repo_commit = None

print(f'In Colab-style runtime: {IN_COLAB}')
print(f'Force repo refresh: {FORCE_REPO_REFRESH}')
print(f'Google Drive mounted: {DRIVE_MOUNTED}')
print(f'Repo root: {REPO_ROOT}')
print(f'Project dir: {PROJECT_DIR}')
print(f'Dataset dir: {DATASET_DIR}')
print(f'Saves dir: {SAVES_DIR}')
if repo_commit:
    print(f'Repo commit: {repo_commit}')


In [ ]:
from pathlib import Path
print(f"Working directory: {Path.cwd()}")


Working directory: /content/560-FaceRecognition/project-fr


In [ ]:
from pathlib import Path
cwd = Path.cwd()
print(cwd)
print("\n".join(sorted(path.name for path in cwd.iterdir())))


/content/560-FaceRecognition/project-fr
README.md
__pycache__
evaluate.py
make_validation_split.py
models
train_example.py


In [ ]:
required_files = [
    TRAIN_DATASET_DIR / "test.parquet",
    TRAIN_DATASET_DIR / "pairs.parquet",
    TRAIN_METADATA_PATH,
    VAL_METADATA_PATH,
    VAL_PAIRS_PATH,
]
for path in required_files:
    print(f"{path.name}: {'found' if path.exists() else 'missing'}")
print(f"Validation split available: {SPLIT_DIR.exists()}")


test.parquet: found
pairs.parquet: found
train_metadata.parquet: found
val_metadata.parquet: found
val_pairs.parquet: found
Validation split available: True


In [ ]:
if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"Images directory not found at {IMAGES_DIR}")

sample_entry = next(IMAGES_DIR.iterdir(), None)
print(f"Images directory ready: {IMAGES_DIR}")
print(f"Sample entry: {sample_entry.name if sample_entry else 'empty directory'}")


Images directory ready: /content/drive/MyDrive/560/Final-Project/560-FaceRecognition/project-fr/datasets/dataset_a/images
Sample entry: 42592.jpg


In [ ]:
from datetime import datetime

log_path = SAVES_DIR / "logs.txt"
with log_path.open("a", encoding="utf-8") as handle:
    handle.write(f"{datetime.now().isoformat()}\n")

print(f"Logging to {log_path}")


Logging to /content/drive/MyDrive/560/Final-Project/560-FaceRecognition/project-fr/saves/logs.txt


# Train Model

In [ ]:
import json
import subprocess
import sys
import time

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    device = "cpu"

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

BACKBONE = "mobilefacenet"
EMBEDDING_DIM = "128"
EPOCHS = "1"
# For a longer baseline run, increase EPOCHS after this smoke test passes.

RUN_SAVE_DIR = SAVES_DIR / f"{BACKBONE}_smoketest_ep{EPOCHS}"
RUN_SAVE_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-u",
    str(PROJECT_DIR / "train_example.py"),
    "--data_root", str(TRAIN_DATASET_DIR),
    "--save_dir", str(RUN_SAVE_DIR),
    "--device", device,
    "--backbone", BACKBONE,
    "--num_workers", "8" if device == "cuda" else "0",
    "--batch_size", "128",
    "--epochs", EPOCHS,
    "--embedding_dim", EMBEDDING_DIM,
    "--lr", "3e-4",
    "--weight_decay", "1e-4",
    "--arcface_m", "0.4",
    "--arcface_s", "48",
]

cmd.extend([
    "--train_metadata", str(TRAIN_METADATA_PATH),
    "--val_metadata", str(VAL_METADATA_PATH),
    "--val_pairs", str(VAL_PAIRS_PATH),
])

help_result = subprocess.run(
    [sys.executable, str(PROJECT_DIR / "train_example.py"), "--help"],
    text=True,
    capture_output=True,
    cwd=str(PROJECT_DIR),
)
help_text = (help_result.stdout or "") + (help_result.stderr or "")

if BACKBONE == "mobilefacenet" and "mobilefacenet" not in help_text:
    raise RuntimeError(
        "The train_example.py visible to this notebook runtime does not support mobilefacenet. "
        "This usually means the notebook is running against an older cloned repo instead of your updated workspace."
    )

if device == "cuda" and "--amp" in help_text:
    cmd.append("--amp")

print("Running:", " ".join(cmd))
start_time = time.time()
result = subprocess.run(cmd, text=True, capture_output=True, cwd=str(PROJECT_DIR))
elapsed_time = time.time() - start_time
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Training command failed with exit code {result.returncode}\n"
        f"STDOUT:\n{result.stdout}\n"
        f"STDERR:\n{result.stderr}"
    )

metrics_path = RUN_SAVE_DIR / "metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(f"Expected metrics file at {metrics_path}")

with metrics_path.open("r", encoding="utf-8") as handle:
    metrics = json.load(handle)

performance = metrics.get("best_val_performance")
if performance is None:
    for entry in reversed(metrics.get("history", [])):
        if entry.get("val_performance") is not None:
            performance = entry["val_performance"]
            break

summary_fields = {
    "training_wall_time_seconds": elapsed_time,
    "total_time_seconds": metrics.get("total_time_seconds", elapsed_time),
    "embedding_size": metrics.get("embedding_size", metrics.get("args", {}).get("embedding_dim")),
    "best_epoch": metrics.get("best_epoch"),
    "throughput_images_per_second": metrics.get("throughput_images_per_second"),
    "train_images_per_second": metrics.get("train_images_per_second"),
    "peak_gpu_memory_allocated_mb": metrics.get("peak_gpu_memory_allocated_mb"),
    "peak_gpu_memory_reserved_mb": metrics.get("peak_gpu_memory_reserved_mb"),
}

def perf_metric(*names):
    for name in names:
        if performance is not None and name in performance:
            return performance.get(name)
    return None

if performance is not None:
    summary_fields.update({
        "AUC": performance.get("AUC"),
        "TAR_FAR_1e_6": perf_metric("TAR@FAR=1e-6", "TAR@FAR=1e-06"),
        "TAR_FAR_1e_5": perf_metric("TAR@FAR=1e-5", "TAR@FAR=1e-05"),
        "TAR_FAR_1e_4": perf_metric("TAR@FAR=1e-4", "TAR@FAR=1e-04"),
        "TAR_FAR_1e_3": perf_metric("TAR@FAR=1e-3", "TAR@FAR=1e-03"),
        "eval_images_per_second": performance.get("eval_images_per_second"),
        "eval_time_seconds": performance.get("eval_time_seconds"),
    })

metrics["colab_summary"] = summary_fields
metrics.update({key: value for key, value in summary_fields.items() if value is not None})

with metrics_path.open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)

required_metric_keys = [
    "AUC",
    "TAR_FAR_1e_6",
    "TAR_FAR_1e_5",
    "TAR_FAR_1e_4",
    "TAR_FAR_1e_3",
]
missing_metric_keys = [key for key in required_metric_keys if metrics.get(key) is None]
if missing_metric_keys:
    raise RuntimeError(
        "Training finished, but metrics.json is still missing required performance metrics: "
        + ", ".join(missing_metric_keys)
    )

print("Saved metrics summary:")
print(f"  TAR@FAR=1e-6: {summary_fields['TAR_FAR_1e_6']:.2f}%")
print(f"  TAR@FAR=1e-5: {summary_fields['TAR_FAR_1e_5']:.2f}%")
print(f"  TAR@FAR=1e-4: {summary_fields['TAR_FAR_1e_4']:.2f}%")
print(f"  TAR@FAR=1e-3: {summary_fields['TAR_FAR_1e_3']:.2f}%")
print(f"  AUC: {summary_fields['AUC']:.2f}%")
if summary_fields.get('throughput_images_per_second') is not None:
    print(f"  Throughput: {summary_fields['throughput_images_per_second']:.2f} images/s")
if summary_fields.get('peak_gpu_memory_reserved_mb') is not None and summary_fields.get('peak_gpu_memory_allocated_mb') is not None:
    print(
        "  Peak GPU memory: "
        f"{summary_fields['peak_gpu_memory_reserved_mb']:.2f} MB reserved "
        f"({summary_fields['peak_gpu_memory_allocated_mb']:.2f} MB allocated)"
    )
print(f"  Embedding size: {summary_fields['embedding_size']}")
print(f"  Total time: {summary_fields['total_time_seconds']:.2f} s")


RuntimeError: The train_example.py visible to this notebook runtime does not support mobilefacenet. This usually means the notebook is running against an older cloned repo instead of your updated workspace.